# Очищення датасету

Цей ноутбук:
- видаляє дублікати
- конвертує значення label з 1 у 0 у базовому файлі
- додає всі рядки з NEWS.csv, де label=1, і конвертує їх у 0
- додає всі рядки з NEWS.csv, де label=0, і конвертує їх у 1
- додає true-новини з news_data.csv (колонки Text, Label) для вирівнювання кількості true/fake
- зберігає результат у новий CSV

In [11]:
import pandas as pd
from pathlib import Path

In [18]:
project_root = Path.cwd()
input_path = project_root / 'Datasets' / 'news_detector' / 'last_final_fake_true1.csv'
news_path = project_root / 'Datasets' / 'news_detector' / 'NEWS.csv'
news_data_path = project_root / 'Datasets' / 'news_detector' / 'news_data.csv'
output_path = project_root / 'Datasets' / 'news_detector' / 'last_final_fake_true1_cleaned.csv'

print(f'Input: {input_path}')
print(f'NEWS source: {news_path}')
print(f'news_data source: {news_data_path}')
print(f'Output: {output_path}')

Input: c:\Users\igrew\OneDrive\Desktop\Course Work\Datasets\news_detector\last_final_fake_true1.csv
NEWS source: c:\Users\igrew\OneDrive\Desktop\Course Work\Datasets\news_detector\NEWS.csv
news_data source: c:\Users\igrew\OneDrive\Desktop\Course Work\Datasets\news_detector\news_data.csv
Output: c:\Users\igrew\OneDrive\Desktop\Course Work\Datasets\news_detector\last_final_fake_true1_cleaned.csv


In [19]:
df = pd.read_csv(input_path)
print('Початкова кількість рядків:', len(df))
if 'label' in df.columns:
    print('Початковий розподіл label:')
    print(df['label'].value_counts(dropna=False))
else:
    raise ValueError("У файлі немає колонки 'label'")

Початкова кількість рядків: 11538
Початковий розподіл label:
label
0    10865
1      673
Name: count, dtype: int64


In [20]:
before_duplicates = len(df)
df = df.drop_duplicates()
removed_duplicates = before_duplicates - len(df)

if 'label' not in df.columns:
    raise ValueError("У вхідному файлі немає колонки 'label'")

if 'text' not in df.columns:
    raise ValueError("У вхідному файлі немає колонки 'text'")

# Базова конвертація: у вхідному файлі 1 -> 0
df['label'] = df['label'].replace(1, 0)

news_df = pd.read_csv(news_path)
if not {'text', 'label'}.issubset(news_df.columns):
    raise ValueError("У NEWS.csv потрібні колонки text і label")

# З NEWS.csv: fake (1) додаємо як 0
news_label_1 = news_df[news_df['label'] == 1][['text', 'label']].copy()
news_label_1['label'] = 0

# З NEWS.csv: true (0) додаємо як 1
news_label_0 = news_df[news_df['label'] == 0][['text', 'label']].copy()
news_label_0['label'] = 1

added_fake_rows = len(news_label_1)
added_true_rows = len(news_label_0)

df = pd.concat([df[['text', 'label']], news_label_1, news_label_0], ignore_index=True)
before_final_dedup = len(df)
df = df.drop_duplicates()
removed_after_merge = before_final_dedup - len(df)

# Додатково: true-новини з news_data.csv (Text, Label) для балансу
news_data_df = pd.read_csv(news_data_path)
if not {'Text', 'Label'}.issubset(news_data_df.columns):
    raise ValueError("У news_data.csv потрібні колонки Text і Label")

news_data_true = news_data_df[news_data_df['Label'] == True][['Text', 'Label']].copy()
news_data_true = news_data_true.rename(columns={'Text': 'text', 'Label': 'label'})
news_data_true['label'] = 1

# Прибираємо порожні та дублікати по тексту у джерелі
news_data_true = news_data_true.dropna(subset=['text'])
news_data_true = news_data_true.drop_duplicates(subset=['text'])

# Беремо тільки ті true-новини, яких ще немає у фінальному df (щоб без дублікатів)
news_data_true = news_data_true[~news_data_true['text'].isin(df['text'])]

fake_count = int((df['label'] == 0).sum())
true_count = int((df['label'] == 1).sum())
needed_for_balance = max(0, fake_count - true_count)

# Орієнтир з задачі: 10192, але пріоритет - зробити рівно true/fake
target_true_addition = 10192
requested_true_addition = needed_for_balance if needed_for_balance > 0 else 0

if requested_true_addition > 0:
    to_add = min(requested_true_addition, len(news_data_true))
    news_data_to_add = news_data_true.head(to_add).copy()
    df = pd.concat([df, news_data_to_add], ignore_index=True)
    df = df.drop_duplicates()
else:
    to_add = 0

final_fake_count = int((df['label'] == 0).sum())
final_true_count = int((df['label'] == 1).sum())

print('Видалено дублікатів у базовому файлі:', removed_duplicates)
print('Додано рядків з NEWS.csv (було label=1 -> стало 0):', added_fake_rows)
print('Додано рядків з NEWS.csv (було label=0 -> стало 1):', added_true_rows)
print('Видалено дублікатів після злиття:', removed_after_merge)
print('Орієнтир додавання true з news_data.csv:', target_true_addition)
print('Потрібно true до балансу:', requested_true_addition)
print('Фактично додано true з news_data.csv:', to_add)
print('Фінальна кількість рядків:', len(df))
print('Фінальний розподіл label:')
print(df['label'].value_counts(dropna=False))
print('Різниця між fake і true:', abs(final_fake_count - final_true_count))

Видалено дублікатів у базовому файлі: 0
Додано рядків з NEWS.csv (було label=1 -> стало 0): 2862
Додано рядків з NEWS.csv (було label=0 -> стало 1): 2022
Видалено дублікатів після злиття: 4880
Орієнтир додавання true з news_data.csv: 10192
Потрібно true до балансу: 10196
Фактично додано true з news_data.csv: 10196
Фінальна кількість рядків: 21738
Фінальний розподіл label:
label
0    10869
1    10869
Name: count, dtype: int64
Різниця між fake і true: 0


In [21]:
df.to_csv(output_path, index=False, encoding='utf-8')
print(f'Збережено: {output_path}')
df.head()

Збережено: c:\Users\igrew\OneDrive\Desktop\Course Work\Datasets\news_detector\last_final_fake_true1_cleaned.csv


,text,label
0,Андріївка – явка з повинною окупантів: дайджес...,0
1,У біолабораторіях України проводили досліди з ...,0
2,Білл Гейтс закликав негайно прибрати з ринку у...,0
3,Королеву Єлизавету II лікують від COVID-19 «за...,0
4,Вакцини проти COVID-19 збільшили дитячу смертн...,0
